# Whirlpool Experiment — Master Runner (Morocco)

Executes the full analysis pipeline without visualizations:

1. **Load run data** — attribute tables + wide-format simulation output
2. **Post-processing** — intertemporal decomposition / rescaling → `decomposed_ssp_output_whirlpool.csv`
3. **Cost-benefits** — system + technical costs → `cost_benefits_data_whirlpool.csv`
4. **MAC analysis** — marginal abatement costs → `marginal_abatement_costs_whirlpool.csv`
5. **Tableau export** — whirlpool plot data → `tableau/data/tableau_whirlpool.csv`
6. **MAC comparison** — tornado vs whirlpool → `tableau/data/mac_tornado_to_whirlpool.csv`

**Whirlpool design.** Every strategy is the full portfolio MINUS one
transformation (`WHIRLPOOL_PFLO_LEDS:TX:<SECTOR>:<NAME>`), so the MAC baseline is
the complete portfolio — **`PFLO:LEDS`** for Morocco (it was `PFLO:CONDITIONAL`
for Libya).  Cost-benefits still runs against `BASE`; `mac_pipeline` re-bases the
technical costs on the portfolio.

**Steps 5-6 need the tornado run.** Run `tornado_runner/run_experiment.ipynb`
first — it produces `marginal_abatement_costs_tornado.csv` and
`ATTRIBUTE_MAP_TORNADO_WHIRLPOOL.csv`, which `config.py` picks up automatically.
Keep `YEAR_CUMUL_START` identical in both configs or the two MACs are not
comparable.

All configuration lives in `scripts/config.py`.  Shared pipeline modules
(`data_loading`, `postprocessing`, `cost_benefits_pipeline`) come from
`../shared_scripts/` — the Morocco versions used by
`workflow/morocco_manager_wb.ipynb`.


## 0 · Environment setup

In [ ]:
import os
import sys
import pathlib
import logging
import warnings

%load_ext autoreload
%autoreload 2

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
RUNNER_DIR  = pathlib.Path(os.getcwd()).resolve()
SCRIPTS_DIR = RUNNER_DIR / "scripts"
SHARED_DIR  = RUNNER_DIR.parent / "shared_scripts"

assert SCRIPTS_DIR.exists(), f"scripts/ not found at {SCRIPTS_DIR}"
assert SHARED_DIR.exists(),  f"shared_scripts/ not found at {SHARED_DIR}"

for p in (SCRIPTS_DIR, SHARED_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# ── Config ────────────────────────────────────────────────────────────────────
import config as cfg

# Project root (needed for ssp_modeling.* imports inside pipeline scripts)
if str(cfg.PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(cfg.PROJECT_DIR))

# notebooks/ dir (needed for utils.logger_utils)
if str(cfg.NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(cfg.NOTEBOOKS_DIR))

from utils.logger_utils import setup_clean_logger, mute_external_loggers

logger = setup_clean_logger("whirlpool", logging.INFO)
mute_external_loggers(["sisepuede"])
logger.info("Environment ready.")
logger.info(f"Project dir   : {cfg.PROJECT_DIR}")
logger.info(f"Run ID        : {cfg.RUN_ID}")
logger.info(f"Run output    : {cfg.RUN_ID_OUTPUT_DIR}")
logger.info(f"Tornado run   : {cfg.TORNADO_RUN_ID}")
logger.info(f"Region / ISO  : {cfg.REGION} / {cfg.ISO_CODE3}")
logger.info(f"Years         : {cfg.YEAR_START}-{cfg.YEAR_END} (ref {cfg.YEAR_REF})")
logger.info(f"MAC baseline  : {cfg.STRATEGY_CODE_PORTFOLIO}")
logger.info(f"Targets       : {cfg.TARGETS_PATH.name}")
logger.info(f"Inventory     : {cfg.INVENT_HISTORIC_PATH.name}")
logger.info(f"Tableau dir   : {cfg.TABLEAU_DIR}")
logger.info(f"Primary IDs   : {len(cfg.PRIMARY_IDS_FILTER)}")

# Fail fast if the run or the reference data are missing
for _label, _path in [
    ("targets",   cfg.TARGETS_PATH),
    ("inventory", cfg.INVENT_HISTORIC_PATH),
    ("cb config", cfg.CB_CONFIG_PATH),
    ("run dir",   cfg.RUN_ID_OUTPUT_DIR),
]:
    assert _path.exists(), f"{_label} not found: {_path}"


## 1 · Load run data

In [ ]:
from data_loading import load_attribute_tables, parse_strategy_metadata, load_wide_export

# Attribute tables
att_primary, att_strategy = load_attribute_tables(cfg.RUN_ID_OUTPUT_DIR)
att_strategy = parse_strategy_metadata(att_strategy)

logger.info(f"att_primary  : {att_primary.shape}")
logger.info(f"att_strategy : {att_strategy.shape}")

# Wide-format simulation output
df_export = load_wide_export(cfg.RUN_ID_OUTPUT_DIR, cfg.PRIMARY_IDS_FILTER)
logger.info(f"df_export    : {df_export.shape}")

## 2 · Post-processing — intertemporal decomposition

In [ ]:
from postprocessing import run_decomposition

df_decomposed = run_decomposition(
    df_export    = df_export,
    project_dir  = cfg.PROJECT_DIR,
    targets_path = cfg.TARGETS_PATH,
    iso_code3    = cfg.ISO_CODE3,
    year_ref     = cfg.YEAR_REF,
    region       = cfg.REGION,
    output_path  = cfg.OUTPUT_DECOMPOSED,
    initial_conditions_id = cfg.INITIAL_CONDITIONS_ID,   # BASE → primary_id 0
)
logger.info(f"df_decomposed : {df_decomposed.shape}  → {cfg.OUTPUT_DECOMPOSED}")

## 3 · Cost-benefits

In [ ]:
from cost_benefits_pipeline import run_cost_benefits

cb_data = run_cost_benefits(
    df_decomposed      = df_decomposed,
    att_primary        = att_primary,
    att_strategy       = att_strategy,
    cb_config_path     = cfg.CB_CONFIG_PATH,
    run_output_dir     = cfg.RUN_ID_OUTPUT_DIR,
    project_dir        = cfg.PROJECT_DIR,
    strategy_code_base = cfg.STRATEGY_CODE_BASE,
    output_path        = cfg.OUTPUT_CB_DATA,
)
logger.info(f"cb_data : {cb_data.shape}  → {cfg.OUTPUT_CB_DATA}")

In [ ]:
# ── Cargar cb_data desde CSV (si ya fue calculado) ───────────────────────────
# Ejecutar esta celda en lugar de la celda de arriba si cb_data ya fue guardado
import pandas as pd

cb_data = pd.read_csv(cfg.OUTPUT_CB_DATA)
logger.info(f"cb_data cargado : {cb_data.shape}  ← {cfg.OUTPUT_CB_DATA}")

## 4 · Marginal Abatement Cost (MAC)

In [ ]:
from mac_pipeline import run_mac_analysis

mac_df = run_mac_analysis(
    df_decomposed           = df_decomposed,
    cb_data                 = cb_data,
    att_primary             = att_primary,
    att_strategy            = att_strategy,
    iso_code3               = cfg.ISO_CODE3,
    region                  = cfg.REGION,
    invent_dir              = cfg.INVENT_DIR,
    invent_historic_path    = cfg.INVENT_HISTORIC_PATH,    # invent_historic_mar.csv
    targets_path            = cfg.TARGETS_PATH,
    run_output_dir          = cfg.RUN_ID_OUTPUT_DIR,
    strategy_code_portfolio = cfg.STRATEGY_CODE_PORTFOLIO, # PFLO:LEDS
    output_filename         = cfg.OUTPUT_MAC.name,
    year_cumul_start        = cfg.YEAR_CUMUL_START,        # None → 2018 (last inventory year)
    year_start              = cfg.YEAR_START,
)
logger.info(f"mac_df : {mac_df.shape}  → {cfg.OUTPUT_MAC}")


## 5 · Tableau export — whirlpool

In [ ]:
from mac_pipeline import build_attribute_map

# Steps 5-6 need the tornado outputs; run tornado_runner/run_experiment.ipynb first.
assert cfg.TORNADO_RUN_ID is not None, (
    "No tornado run found. Run tornado_runner/run_experiment.ipynb, or pin "
    "TORNADO_RUN_ID in scripts/config.py."
)
assert cfg.TORNADO_ATT_MAP_PATH.exists(), f"missing: {cfg.TORNADO_ATT_MAP_PATH}"
assert cfg.TORNADO_MAC_PATH.exists(),     f"missing: {cfg.TORNADO_MAC_PATH}"

att_map = build_attribute_map(
    att_primary          = att_primary,
    att_strategy         = att_strategy,
    run_output_dir       = cfg.RUN_ID_OUTPUT_DIR,
    tornado_att_map_path = cfg.TORNADO_ATT_MAP_PATH,
)
logger.info(f"att_map : {att_map.shape}  → {cfg.RUN_ID_OUTPUT_DIR / 'ATTRIBUTE_MAP_TORNADO_WHIRLPOOL.csv'}")

# Every whirlpool strategy that removes one transformation should find its
# tornado twin.  The portfolio baseline row (PFLO:LEDS) has no
# transformation_code and is expected to stay unmatched.
singles   = att_map[att_map["transformation_code"].notna()]
unmatched = singles[singles["primary_id_tornado"].isna()]
logger.info(f"matched tornado↔whirlpool: {len(singles) - len(unmatched)}/{len(singles)}")
if len(unmatched):
    logger.warning(f"{len(unmatched)} whirlpool strategies have no tornado match:")
    print(unmatched[["primary_id_whirlpool", "transformation_code", "sector"]].to_string(index=False))


In [ ]:
from mac_pipeline import build_tableau_whirlpool

# Exclude full-portfolio rows (PFLO:LEDS itself) — keep only individual transformations
mac_df = mac_df[mac_df["transformation_code"].notna() & (mac_df["transformation_code"] != "")]

tableau_whirlpool = build_tableau_whirlpool(
    mac_df           = mac_df,
    run_output_dir   = cfg.RUN_ID_OUTPUT_DIR,
    tableau_dir      = cfg.TABLEAU_DIR,
    tornado_mac_path = cfg.TORNADO_MAC_PATH,
    output_path      = cfg.OUTPUT_TABLEAU_WHIRLPOOL,
)
logger.info(f"tableau_whirlpool : {tableau_whirlpool.shape}  → {cfg.OUTPUT_TABLEAU_WHIRLPOOL}")


## 6 · MAC tornado vs whirlpool

In [ ]:
from mac_pipeline import build_mac_tornado_to_whirlpool

mac_comparison = build_mac_tornado_to_whirlpool(
    whirlpool_mac_df = mac_df,
    run_output_dir   = cfg.RUN_ID_OUTPUT_DIR,
    tableau_dir      = cfg.TABLEAU_DIR,
    tornado_mac_path = cfg.TORNADO_MAC_PATH,
    output_path      = cfg.OUTPUT_MAC_TORNADO_TO_WHIRLPOOL,
)
logger.info(f"mac_comparison : {mac_comparison.shape}  → {cfg.OUTPUT_MAC_TORNADO_TO_WHIRLPOOL}")

## 7 · Summary

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {"output": "decomposed SSP",     "rows": len(df_decomposed),     "cols": df_decomposed.shape[1],     "file": str(cfg.OUTPUT_DECOMPOSED)},
    {"output": "cost-benefits data", "rows": len(cb_data),           "cols": cb_data.shape[1],           "file": str(cfg.OUTPUT_CB_DATA)},
    {"output": "MAC curves",         "rows": len(mac_df),            "cols": mac_df.shape[1],            "file": str(cfg.OUTPUT_MAC)},
    {"output": "tableau whirlpool",  "rows": len(tableau_whirlpool), "cols": tableau_whirlpool.shape[1], "file": str(cfg.OUTPUT_TABLEAU_WHIRLPOOL)},
    {"output": "MAC tornado↔whirl",  "rows": len(mac_comparison),    "cols": mac_comparison.shape[1],    "file": str(cfg.OUTPUT_MAC_TORNADO_TO_WHIRLPOOL)},
])
print(summary.to_string(index=False))

# Tornado vs whirlpool: large gaps flag interactions between transformations
cmp = mac_comparison.dropna(subset=["mac_tornado", "mac_whirlpool"]).copy()
cmp["mac_gap"] = cmp["mac_whirlpool"] - cmp["mac_tornado"]
print("\nLargest tornado↔whirlpool MAC gaps (interaction effects):")
print(
    cmp.reindex(cmp["mac_gap"].abs().sort_values(ascending=False).index)
    .head(10)[["transformation_name_sector", "mac_tornado", "mac_whirlpool", "mac_gap"]]
    .to_string(index=False)
)
